# Downbeat Snapping Demo

The CRNN predicts chorus probabilities on a meter grid extrapolated from a single global tempo estimate. That grid has two weaknesses: its phase is anchored to the first detected *beat* (which may not be a downbeat), and a constant tempo drifts over the length of a song. Both push predicted boundaries a beat or a bar off the true downbeats.

This notebook shows the post-processing fix: track downbeats with [Beat This!](https://github.com/CPJKU/beat_this) (CPJKU, ISMIR 2024) and snap each chorus boundary to the downbeat within a ±2-bar window that shows the strongest RMS energy rise (starts) or fall (ends).

It demonstrates the same functionality exposed by the CLI scripts:

| Script | Purpose |
|---|---|
| `scripts/inference.py` | Detect choruses (snapping on by default, `--no-snap` to disable, `--snap-window N` to widen/narrow the search) |
| `scripts/visualize_snap.py` | Plot raw vs snapped boundaries on the waveform with the downbeat grid |
| `scripts/evaluate_snap.py` | Measure shift and ground-truth error across the labeled dataset |

## 1. Setup

Requires the repo environment (`requirements.txt`, which includes `beat_this`) and a model checkpoint. The pretrained model downloads automatically if missing.

In [1]:
import os
import sys

import librosa
import numpy as np
import torch

sys.path.append(os.path.abspath(".."))

from pytorch_core.audio_processor import process_audio
from pytorch_core.downbeats import snap_chorus_segments, track_downbeats
from pytorch_core.model import download_model, load_CRNN_model, smooth_predictions, MODEL_PATH
from pytorch_core.visualization import plot_snap_comparison

# Change this to any song you want to inspect
AUDIO_PATH = "../data/audio/processed/1.mp3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = load_CRNN_model(MODEL_PATH)
print(f"model loaded, device for downbeat tracking: {DEVICE}")

c:\Users\denni\OneDrive\Desktop\chorus-detection\pytorch_core\model.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location="cp

model loaded, device for downbeat tracking: cuda


## 2. Raw chorus boundaries from the meter grid

Run the standard pipeline: feature extraction, meter-grid segmentation, CRNN prediction, smoothing, and grouping of consecutive positive meters into chorus segments.

Note `trim_silence=False` here because the processed dataset audio is already trimmed; for arbitrary files (as in `scripts/inference.py`) use `trim_silence=True`.

In [ ]:
processed_audio, audio_features = process_audio(AUDIO_PATH, trim_silence=False)

with torch.no_grad():
    outputs = model(torch.tensor(processed_audio, dtype=torch.float32)).numpy().squeeze()

n_meters = min(len(audio_features.meter_grid) - 1, len(outputs))
smoothed = smooth_predictions(outputs[:n_meters])

meter_grid_times = librosa.frames_to_time(
    audio_features.meter_grid, sr=audio_features.sr, hop_length=audio_features.hop_length)

# Group consecutive positive meters into segments
raw_starts, raw_ends = [], []
indices = np.where(smoothed == 1)[0]
group = [indices[0]]
for i in indices[1:]:
    if i == group[-1] + 1:
        group.append(i)
    else:
        raw_starts.append(meter_grid_times[group[0]])
        raw_ends.append(meter_grid_times[group[-1] + 1])
        group = [i]
raw_starts.append(meter_grid_times[group[0]])
raw_ends.append(meter_grid_times[group[-1] + 1])

for i, (s, e) in enumerate(zip(raw_starts, raw_ends)):
    print(f"raw chorus {i+1}: {s:7.2f}s - {e:7.2f}s")

## 3. Track downbeats with Beat This!

`track_downbeats()` returns beat and downbeat times in seconds. The tracker checkpoint (`final0`) downloads on first use. The median gap between downbeats is the bar length, which sizes the snap search window.

In [ ]:
beats, downbeats = track_downbeats(AUDIO_PATH, device=DEVICE)
bar_length = np.median(np.diff(downbeats))
print(f"{len(beats)} beats, {len(downbeats)} downbeats, median bar {bar_length:.2f}s")
print("first downbeats:", np.round(downbeats[:8], 2))

## 4. Snap boundaries to downbeats

With an energy envelope, each start snaps to the downbeat within ±2 bars with the strongest energy **rise** (a drop/chorus onset) and each end to the strongest **fall**. Without energy it falls back to the nearest downbeat. Segments that collapse are dropped; overlapping ones merge.

In [ ]:
rms = np.asarray(audio_features.rms).ravel()
rms_times = librosa.frames_to_time(
    np.arange(rms.size), sr=audio_features.sr, hop_length=audio_features.hop_length)

snapped_starts, snapped_ends = snap_chorus_segments(
    raw_starts, raw_ends, downbeats,
    energy=rms, energy_times=rms_times, search_bars=2.0)

print(f"{'':>10} {'raw':>8} {'snapped':>8} {'shift':>7}")
for i, (rs, ss) in enumerate(zip(raw_starts, snapped_starts)):
    print(f"start {i+1:>3}: {rs:8.2f} {ss:8.2f} {ss-rs:+7.2f}")
for i, (re_, se) in enumerate(zip(raw_ends, snapped_ends)):
    print(f"end   {i+1:>3}: {re_:8.2f} {se:8.2f} {se-re_:+7.2f}")

## 5. Visualize raw vs snapped

Full waveform with the downbeat grid (grey dashed), raw boundaries (red dashed) and snapped boundaries (green solid), plus a zoom panel around every snapped boundary.

CLI equivalent:
```bash
python scripts/visualize_snap.py --audio data/audio/processed/1.mp3 \
    --checkpoint models/pytorch/best_model.pt --output output/snap_comparison.png
```

In [ ]:
fig = plot_snap_comparison(audio_features, raw_starts, raw_ends,
                           snapped_starts, snapped_ends, downbeats)

## 6. Evaluate across the labeled dataset

`scripts/evaluate_snap.py` sweeps every labeled song, writing one row per boundary with the snap shift and the error of raw/snapped boundaries against the annotated ground truth. It appends incrementally, so an interrupted run resumes where it left off.

```bash
python scripts/evaluate_snap.py --checkpoint models/pytorch/best_model.pt \
    --device cuda --output output/snap_evaluation.csv
# re-print the summary of an existing results file:
python scripts/evaluate_snap.py --checkpoint models/pytorch/best_model.pt \
    --output output/snap_evaluation.csv --summary-only
```

The cells below analyze the results CSV directly.

In [ ]:
import pandas as pd

df = pd.read_csv("../output/snap_evaluation.csv")
print(f"{df['song_id'].nunique()} songs, {len(df)} boundaries")
df.head()

In [ ]:
# How far and in which direction did snapping move the boundaries?
shift = df["shift"]
print(f"shift (snapped - raw): mean {shift.mean():+.3f}s, median {shift.median():+.3f}s")
print(f"absolute shift:        mean {shift.abs().mean():.3f}s, median {shift.abs().median():.3f}s")
print(f"moved later {(shift > 0).mean():.1%}, earlier {(shift < 0).mean():.1%}, unmoved {(shift == 0).mean():.1%}")

In [ ]:
# Did snapping bring boundaries closer to the annotated truth?
scored = df.dropna(subset=["raw_error", "snapped_error"])
for kind in ("start", "end"):
    sub = scored[scored["kind"] == kind]
    raw_abs, snap_abs, bars = sub["raw_error"].abs(), sub["snapped_error"].abs(), sub["bar_length"]
    print(f"{kind:>5}: raw median |err| {raw_abs.median():.2f}s -> snapped {snap_abs.median():.2f}s | "
          f"within 1 bar: {(raw_abs <= bars).mean():.1%} -> {(snap_abs <= bars).mean():.1%} | "
          f"within 0.5s: {(raw_abs <= 0.5).mean():.1%} -> {(snap_abs <= 0.5).mean():.1%}")

signed = scored["snapped_error"]
print(f"\nsnapped signed error vs truth: median {signed.median():+.3f}s "
      f"(late {(signed > 0).mean():.1%} / early {(signed < 0).mean():.1%})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(shift.clip(-6, 6), bins=49, color="steelblue")
axes[0].set_title("Snap shift (snapped - raw), s")
axes[0].axvline(0, color="k", linewidth=0.8)

axes[1].hist(scored["raw_error"].clip(-8, 8), bins=49, alpha=0.6, label="raw", color="red")
axes[1].hist(scored["snapped_error"].clip(-8, 8), bins=49, alpha=0.6, label="snapped", color="green")
axes[1].set_title("Signed error vs ground truth, s")
axes[1].axvline(0, color="k", linewidth=0.8)
axes[1].legend()

err_bars = scored["snapped_error"].abs() / scored["bar_length"]
axes[2].hist(err_bars.clip(0, 4), bins=40, color="seagreen")
axes[2].set_title("Snapped |error| in bars")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()

## Notes

- Snapping is **on by default** in `scripts/inference.py` and the web app; it degrades gracefully (unsnapped boundaries, with a console note) when `beat_this` is not installed or the tracker fails.
- `search_bars` (CLI: `--snap-window`) is the half-width of the energy search window in bars. Larger windows can fix bigger grid errors but risk jumping to a neighboring section's energy change.
- Next steps from the research plan: phrase-grid anchoring (8/16/32-bar hypermeasure counting from the first drop), segment re-scoring against `onset+4`/`onset+8` bars to remove pre-chorus contamination, and a duration-grammar DP pass over the per-meter probabilities.